In [1]:
import asyncio
import itertools
from typing import List
from benchmarks import benchmark_orchestrator
from benchmarks.answer_generators import (
    AdkAnswerGenerator,
    GeminiAnswerGenerator,
    GroundTruthAnswerGenerator,
    TrivialAnswerGenerator,
)
from benchmarks.data_models import BenchmarkRunResult
import pandas as pd
from pathlib import Path

# Set pandas display options
pd.set_option('display.max_colwidth', None)

# ANSI escape codes for colors
class bcolors:
    HEADER = '\033[95m'
    OKBLUE = '\033[94m'
    OKCYAN = '\033[96m'
    OKGREEN = '\033[92m'
    WARNING = '\033[93m'
    FAIL = '\033[91m'
    ENDC = '\033[0m'
    BOLD = '\033[1m'
    UNDERLINE = '\033[4m'

def permute(cls, **kwargs):
    """Helper to generate permutations of class instances."""
    keys = kwargs.keys()
    values = kwargs.values()
    for instance_values in itertools.product(*values):
        yield cls(**dict(zip(keys, instance_values)))

# Read context from llms.txt
# try:
#     with open("llms.txt", "r", encoding="utf-8") as f:
#         llms_context = f.read()
# except FileNotFoundError:
#     print(f"{bcolors.WARNING}Warning: llms.txt not found. Proceeding without context.{bcolors.ENDC}")
#     llms_context = ""

# # Read context from llms-relevant.txt
# try:
#     with open("llms-relevant.txt", "r", encoding="utf-8") as f:
#         llms_context_relevant = f.read()
# except FileNotFoundError:
#     print(f"{bcolors.WARNING}Warning: llms.txt not found. Proceeding without context.{bcolors.ENDC}")
#     llms_context_relevant = ""    

async def run_comparison() -> List[BenchmarkRunResult]:
    """Sets up and runs the benchmark comparison."""
    print("Configuring benchmark run...")
    
    benchmark_suites = [
        "benchmarks/benchmark_definitions/api_understanding/benchmark.yaml",
        "benchmarks/benchmark_definitions/fix_errors/benchmark.yaml",
        "benchmarks/benchmark_definitions/diagnose_setup_errors_mc/benchmark.yaml",
        "benchmarks/benchmark_definitions/configure_adk_features_mc/benchmark.yaml",
        "benchmarks/benchmark_definitions/predict_runtime_behavior_mc/benchmark.yaml",
    ]
    
    answer_generators = [
        GroundTruthAnswerGenerator(),
        TrivialAnswerGenerator(),
        *permute(
            GeminiAnswerGenerator,
            model_name=["gemini-2.5-flash"],
            context=[None, Path("llms.txt"), Path("llms-relevant.txt")],
        ),
        # AdkAnswerGenerator(),
    ]
    
    print("Executing benchmarks...")
    results = await benchmark_orchestrator.run_benchmarks(
        benchmark_suites=benchmark_suites, 
        answer_generators=answer_generators,
        max_concurrency=30
    )
    
    return results

def analyze_logs(
    results_df: pd.DataFrame, generator_name: str, result_type: str = 'fail'
) -> None:
    """Filters and displays benchmark results for a specific generator and result type."""
    
    result_value = 1 if result_type.lower() == 'pass' else 0
    
    print(f"{bcolors.HEADER}--- Analyzing {result_type.upper()}S for {generator_name} ---{bcolors.ENDC}")
    
    filtered_df = results_df[
        (results_df['answer_generator'] == generator_name) & 
        (results_df['result'] == result_value)
    ]
    
    if filtered_df.empty:
        print(f"{bcolors.OKGREEN}No {result_type}s found for {generator_name}.{bcolors.ENDC}")
        return
    
    for _, row in filtered_df.iterrows():
        print(f"{bcolors.WARNING}Suite: {row['suite']}{bcolors.ENDC}")
        print(f"{bcolors.WARNING}Benchmark: {row['benchmark_name']}{bcolors.ENDC}")
        print(f"{bcolors.OKCYAN}  Answer:{bcolors.ENDC}\n    {row['answer']}")
        if result_type.lower() == 'fail':
            print(f"{bcolors.FAIL}  Validation Error:{bcolors.ENDC}\n    {row['validation_error']}")
            if "temp_test_file" in row and pd.notna(row["temp_test_file"]):
                print(f"{bcolors.OKBLUE}  Temp File:{bcolors.ENDC} {row['temp_test_file']}")
        print("-" * 40)


In [2]:
# Execute the benchmarks
results = await run_comparison()
raw_results_df = pd.DataFrame([r.model_dump() for r in results])

Configuring benchmark run...
Executing benchmarks...
--- Loading benchmark suite: benchmarks/benchmark_definitions/api_understanding/benchmark.yaml ---
  - Queuing tests for answer generator: GroundTruthAnswerGenerator
  - Queuing tests for answer generator: TrivialAnswerGenerator
  - Queuing tests for answer generator: GeminiAnswerGenerator(gemini-2.5-flash)
  - Queuing tests for answer generator: GeminiAnswerGenerator(gemini-2.5-flash)-with-context-llms.txt
  - Queuing tests for answer generator: GeminiAnswerGenerator(gemini-2.5-flash)-with-context-llms-relevant.txt
--- Loading benchmark suite: benchmarks/benchmark_definitions/fix_errors/benchmark.yaml ---
  - Queuing tests for answer generator: GroundTruthAnswerGenerator
  - Queuing tests for answer generator: TrivialAnswerGenerator
  - Queuing tests for answer generator: GeminiAnswerGenerator(gemini-2.5-flash)
  - Queuing tests for answer generator: GeminiAnswerGenerator(gemini-2.5-flash)-with-context-llms.txt
  - Queuing tests for

  0%|                                                                         | 0/985 [00:00<?, ?it/s]

--- PROMPT SENT TO GEMINI ---
You are an expert on the Google ADK Python framework. Answer the following multiple choice question. Return the result as a JSON object with a key 'answer' containing the single letter of the correct option (e.g., 'A', 'B', 'C', or 'D') and a key 'rationale' explaining your reasoning.

Code:
```python
# Copyright 2025 Google LLC
#
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
#     http://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

"""
Test file containing code snippets for 'predict_runtime_behavior_mc' benchmarks.

Each t

  0%|                                                                         | 0/985 [00:02<?, ?it/s]

--- RAW RESPONSE FROM GEMINI ---
{"rationale": "In the Google ADK framework, the `LlmAgent` provides the `save_to_state` parameter specifically for automatically saving the agent's output to the session state. When set to `True` (or a list of specific outputs), the agent's final output will be stored in the session state, making it accessible to subsequent steps or agents in the workflow. The other options do not directly correspond to this functionality within the `LlmAgent` context.", "answer": "A"}
---------------------------------
--- PROMPT SENT TO GEMINI ---
You are an expert on the Google ADK Python framework. Answer the following multiple choice question. Return the result as a JSON object with a key 'answer' containing the single letter of the correct option (e.g., 'A', 'B', 'C', or 'D') and a key 'rationale' explaining your reasoning.

Context:
# Google ADK Python: Definitive API Reference & Developer Guide

This document provides a comprehensive technical reference for the G

AttributeError: 'NoneType' object has no attribute 'log_test_result'

--- RAW RESPONSE FROM GEMINI ---
{
  "rationale": "The question asks about the persistence of `InMemorySessionService`. While `InMemorySessionService` is not explicitly defined in the provided text, the `InMemoryRunner` is described in section '1.3. Runner' as storing its 'state in RAM (lost on exit)'. The term 'In-Memory' universally implies that data is held in the computer's volatile memory (RAM) and is not persistently stored, meaning it will be lost when the program or process terminates. Therefore, `InMemorySessionService` would also store its session data in RAM, making it ephemeral.",
  "answer": "B"
}
---------------------------------
--- RAW RESPONSE FROM GEMINI ---
{
  "rationale": "In the Google ADK Python framework, a `LoggingPlugin` is specifically designed for handling agent logs and directing them to various destinations. If a pre-built `LoggingPlugin` doesn't directly support the desired external monitoring service, the standard approach to extend ADK functionality is 

In [ ]:
raw_results_df.columns

In [ ]:
raw_results_df["suite"] = raw_results_df["suite"].apply(lambda x: x.split("/")[-2])

In [ ]:
# Calculate summary from raw results
summary_df = (
    raw_results_df.groupby(["answer_generator", "suite"])  # Pass columns as a list
    .agg(
        passed=("result", "sum"),
        total=("result", "count"),
        # mean_latency=("latency", "mean"),
        # p50_latency=("latency", lambda x: x.quantile(0.5)),
        # p90_latency=("latency", lambda x: x.quantile(0.9)),
        # p99_latency=("latency", lambda x: x.quantile(0.99)),
    )
)

summary_df["pass_rate"] = summary_df["passed"] / summary_df["total"]

print(f"{bcolors.HEADER}--- Benchmark Summary ---{bcolors.ENDC}")
print(summary_df)

In [ ]:
# --- Analysis Configuration ---
generator_to_analyze = 'GeminiAnswerGenerator(gemini-2.5-pro-with-context)' 
result_type_to_see = 'fail' 

# analyze_logs(
#     results_df=raw_results_df,
#     generator_name=generator_to_analyze,
#     result_type=result_type_to_see
# )
